In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import random

import os, shutil, subprocess, json, sys
from importlib import reload
from joblib import Parallel, delayed

from IPython.display import display

import msprime, tskit

from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import sims
import em_alg

# Visualization defaults
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams['figure.figsize'] = (10, 6)

Установлено NUMBA_NUM_THREADS=32


In [36]:
# parameters for simulation
params = {
    'chrom_length': 5e7,      # DNA sequence length in base pairs
    'recomb_rate': 1e-8,      # Recombination rate (per bp per generation)
    'mut_rate': 2e-8,      # Mutation rate (per bp per generation)
    'gen_time': 29.0,         # Years per generation
    'n_eu': 1,          # European samples
    'n_af': 200,         # African samples
    'n_nd': 10,      # Neanderthal sample number
    'ploidy' : 2
}

# effective pop sizes in demography
Ne = {
    'anc': 18500,      # Common AMH-Neanderthal ancestor
    'nd': 3400,     # Neanderthal population
    'amh': 23000,           # Anatomically modern humans (pre-OoA)
    'ooa': 1861,            # Out-of-Africa founders
    'af': 27600,       # African population
    'eu': 13377,      # European population
    'eu_bottleneck': 1080,  # European bottleneck size
    'eu_growth': 1450       # Post-bottleneck size
}

# split and migration times
# в поколениях
t = {
    't_nd_old_migration': 40000 / params['gen_time'],  # original, older
    't_nd_migration': 30000 / params['gen_time'],  # new_one, younger
    't_amh': 550000 / params['gen_time'],  # AMH-Neanderthal split  
    't_ooa': 73900 / params['gen_time'],   # Out-of-Africa migration
    't_nd_samples': 38000 / params['gen_time'],  # Neanderthal sampling time
    't_eu_growth': 31900 / params['gen_time'],  # European growth start
}

p_admix = {"old": 0.05, "young": 0.03}

OUTPUT_DIR = "sims"

# Проверяем, существует ли папка
if os.path.exists(OUTPUT_DIR):
    print(f"DIR '{OUTPUT_DIR}' exists already. Clean up old files...")
    shutil.rmtree(OUTPUT_DIR) # Удаляет папку рекурсивно вместе со всеми файлами
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_CHR, N_JOBS = 10, 4  # number of chr, -1 use all cores, N use N cores
random_seeds = random.sample(range(1, 10**9), k=N_CHR)

DIR 'sims' exists already. Clean up old files...


In [ ]:
def generate_param_grid():
    p_vals = [0.3, 0.4, 0.5]
    delta_vals = [10000, 15000, 20000]  
    young_vals = [30000, 35000, 40000, 45000, 50000]

    grid = []
    for p_old in p_vals:
        for p_young in p_vals:
            for delta in delta_vals:
                for t_young_kya in young_vals:
                    t_old_kya = t_young_kya + delta
                    if t_old_kya <= 60000:
                        grid.append({
                            "p_old": p_old,
                            "p_young": p_young,
                            "t_young_kya": t_young_kya,
                            "t_old_kya": t_old_kya,
                            "delta_kya": delta
                        })
    return grid

In [ ]:
def run_one_experiment(exp_cfg, base_params, base_Ne, base_t, n_chromosomes=10, n_jobs=4, root_dir="grid_runs"):
    exp_name = (
        f"pold_{exp_cfg['p_old']:.1f}_"
        f"pyoung_{exp_cfg['p_young']:.1f}_"
        f"told_{exp_cfg['t_old_kya']}k_"
        f"tyoung_{exp_cfg['t_young_kya']}k"
    )
    out_dir = os.path.join(root_dir, exp_name)
    os.makedirs(out_dir, exist_ok=True)

    params = dict(base_params)
    Ne = dict(base_Ne)
    t = dict(base_t)

    t["t_nd_old_migration"] = exp_cfg["t_old_kya"] / params["gen_time"]
    t["t_nd_migration"] = exp_cfg["t_young_kya"] / params["gen_time"]

    p_admix = {
        "old": exp_cfg["p_old"],
        "young": exp_cfg["p_young"]
    }

    random.seed(42)
    random_seeds = random.sample(range(1, 10**9), k=n_chromosomes)

    # --- simulate ---
    print("Начались симуляции...")
    gt_parts = Parallel(n_jobs=n_jobs)(
        delayed(sims.process_one_chromosome)(seed, params, Ne, t, p_admix, out_dir)
        for seed in random_seeds
    )
    print("Закончились симуляции.")
    gt_df = pd.concat(gt_parts, ignore_index=True)
    gt_path = os.path.join(out_dir, "all.tracts.summary.csv")
    gt_df.to_csv(gt_path, sep="\t", index=False)

    # --- without EM ---
    print("Начался запуск WITHOUT EM...")
    pred_parts = Parallel(n_jobs=n_jobs)(
        delayed(sims.run_daiseg_task)(seed, out_dir)
        for seed in random_seeds
    )
    print("Закончился запуск WITHOUT EM.")
    pred_parts = [x for x in pred_parts if x is not None]
    if pred_parts:
        pred_df = pd.concat(pred_parts, ignore_index=True)
    else:
        pred_df = pd.DataFrame(columns=["Sample", "CHR", "Start", "End", "Length", "State"])

    pred_path = os.path.join(out_dir, "all.inferred.tracts.tsv")
    pred_df.to_csv(pred_path, sep="\t", index=False)

    metrics_noem = sims.calculate_class_metrics(gt_path, pred_path, int(params["chrom_length"]))
    cm_noem, labels = sims.build_confusion_matrix(gt_df, pred_df, chrom_length=int(params["chrom_length"]), window_size=1000)

    print("\nWITHOUT EM — метрики по классам")
    metrics_noem_df = pd.DataFrame(metrics_noem["Per_class"]).T
    display(metrics_noem_df[["Precision", "Recall", "F1"]])

    print("\nWITHOUT EM — confusion matrix")
    cm_noem_df = pd.DataFrame(cm_noem, index=labels, columns=labels)
    display(cm_noem_df)

    metrics_noem_df.to_csv(os.path.join(out_dir, "metrics_noem.tsv"), sep="\t")
    cm_noem_df.to_csv(os.path.join(out_dir, "cm_noem.tsv"), sep="\t")

    # --- with EM ---
    print("Начался запуск WITH EM...")
    json_files = sorted(str(p) for p in Path(out_dir).glob("config_seed_*.json"))
    em_out_path = os.path.join(out_dir, "all.inferred.EM.tsv")
    em_alg.run_batch_em_pipeline(
        json_files,
        output_combined_file=em_out_path,
        max_iter=15,
        tol=1e-6
    )
    print("Закончился запуск WITH EM.")

    em_df = pd.read_csv(em_out_path, sep="\t") if os.path.exists(em_out_path) else pd.DataFrame(
        columns=["Sample", "CHR", "Start", "End", "Length", "State"]
    )

    metrics_em = sims.calculate_class_metrics(gt_path, em_out_path, int(params["chrom_length"]))
    cm_em, _ = sims.build_confusion_matrix(gt_df, em_df, chrom_length=int(params["chrom_length"]), window_size=1000)

    print("\nWITH EM — метрики по классам")
    metrics_em_df = pd.DataFrame(metrics_em["Per_class"]).T
    display(metrics_em_df[["Precision", "Recall", "F1"]])

    print("\nWITH EM — confusion matrix")
    cm_em_df = pd.DataFrame(cm_em, index=labels, columns=labels)
    display(cm_em_df)

    metrics_em_df.to_csv(os.path.join(out_dir, "metrics_em.tsv"), sep="\t")
    cm_em_df.to_csv(os.path.join(out_dir, "cm_em.tsv"), sep="\t")

    # --- read learned params from file ---
    learned_params_path = em_out_path.replace(".tsv", ".learned_params.json")
    if os.path.exists(learned_params_path):
        with open(learned_params_path, "r", encoding="utf-8") as f:
            learned_params = json.load(f)
    else:
        learned_params = {
            "lambdas": [np.nan, np.nan, np.nan, np.nan],
            "A": None,
            "pi": None,
            "final_log_likelihood": np.nan
        }

    # --- save summary json ---
    summary = {
        "experiment": exp_cfg,
        "without_em": {
            "metrics": metrics_noem,
            "confusion_labels": labels,
            "confusion_matrix": cm_noem.tolist()
        },
        "with_em": {
            "metrics": metrics_em,
            "confusion_labels": labels,
            "confusion_matrix": cm_em.tolist(),
            "learned_lambdas": learned_params.get("lambdas"),
            "learned_A": learned_params.get("A"),
            "learned_pi": learned_params.get("pi"),
            "final_log_likelihood": learned_params.get("final_log_likelihood")
        }
    }

    summary_path = os.path.join(out_dir, "summary.json")
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    row = {
        **exp_cfg,
        "noem_old_precision": metrics_noem["Per_class"].get("Archaic_old", {}).get("Precision", np.nan),
        "noem_old_recall": metrics_noem["Per_class"].get("Archaic_old", {}).get("Recall", np.nan),
        "noem_old_f1": metrics_noem["Per_class"].get("Archaic_old", {}).get("F1", np.nan),
        "noem_young_precision": metrics_noem["Per_class"].get("Archaic_young", {}).get("Precision", np.nan),
        "noem_young_recall": metrics_noem["Per_class"].get("Archaic_young", {}).get("Recall", np.nan),
        "noem_young_f1": metrics_noem["Per_class"].get("Archaic_young", {}).get("F1", np.nan),

        "em_old_precision": metrics_em["Per_class"].get("Archaic_old", {}).get("Precision", np.nan),
        "em_old_recall": metrics_em["Per_class"].get("Archaic_old", {}).get("Recall", np.nan),
        "em_old_f1": metrics_em["Per_class"].get("Archaic_old", {}).get("F1", np.nan),
        "em_young_precision": metrics_em["Per_class"].get("Archaic_young", {}).get("Precision", np.nan),
        "em_young_recall": metrics_em["Per_class"].get("Archaic_young", {}).get("Recall", np.nan),
        "em_young_f1": metrics_em["Per_class"].get("Archaic_young", {}).get("F1", np.nan),

        "learned_lambda_n": summary["with_em"]["learned_lambdas"][0],
        "learned_lambda_af": summary["with_em"]["learned_lambdas"][1],
        "learned_lambda_old": summary["with_em"]["learned_lambdas"][2],
        "learned_lambda_young": summary["with_em"]["learned_lambdas"][3],
        "summary_json": summary_path,
        "out_dir": out_dir
    }

    print("=" * 80)
    
    return row

In [ ]:
grid = generate_param_grid()
all_rows = Parallel(n_jobs=4)(
    delayed(run_one_experiment)(
        cfg,
        base_params=params,
        base_Ne=Ne,
        base_t=t,
        n_chromosomes=10,
        n_jobs=1,              
        root_dir="grid_runs"
    )
    for cfg in grid
)

os.makedirs("grid_runs", exist_ok=True)
results_df = pd.DataFrame(all_rows) # type: ignore
results_df.to_csv("grid_runs/summary_table.tsv", sep="\t", index=False)
results_df

RUN: {'p_old': 0.1, 'p_young': 0.1, 't_young_kya': 30000, 't_old_kya': 35000, 'delta_kya': 5000}
Начались симуляции...
Закончились симуляции.
Начался запуск WITHOUT EM...
Закончился запуск WITHOUT EM.

WITHOUT EM — метрики по классам


,Precision,Recall,F1
Archaic_old,0.5029,0.7774,0.6107
Archaic_young,0.4881,0.2123,0.2959



WITHOUT EM — confusion matrix


,Modern,Archaic_old,Archaic_young
Modern,811977,1258,270
Archaic_old,1641,73568,19841
Archaic_young,1175,70940,19330


Начался запуск WITH EM...
Loading 10 files for Global EM & Inference...
 [obs.py] Loading BED windows...
[obs.py] Loaded 50000 windows. Processing TSV...
   📊 Processed lines: 100000...
 [obs.py] Processing done. Aggregating results...
 Observation sequences for HMM created successfully!
Max differences in 1000bp window: 13
 [obs.py] Loading BED windows...
[obs.py] Loaded 50000 windows. Processing TSV...
   📊 Processed lines: 100000...
 [obs.py] Processing done. Aggregating results...
 Observation sequences for HMM created successfully!
Max differences in 1000bp window: 12
 [obs.py] Loading BED windows...
[obs.py] Loaded 50000 windows. Processing TSV...
   📊 Processed lines: 100000...
 [obs.py] Processing done. Aggregating results...
 Observation sequences for HMM created successfully!
Max differences in 1000bp window: 14
 [obs.py] Loading BED windows...
[obs.py] Loaded 50000 windows. Processing TSV...
   📊 Processed lines: 100000...
 [obs.py] Processing done. Aggregating results...
 O

...


WITH EM — confusion matrix


,Modern,Archaic_old,Archaic_young
Modern,556934,1123,2094
Archaic_old,3275,70777,160140
Archaic_young,1803,41195,162659


RUN: {'p_old': 0.3, 'p_young': 0.2, 't_young_kya': 40000, 't_old_kya': 50000, 'delta_kya': 10000}
Начались симуляции...


/Users/dana/Desktop/курсач 3 курс/DAIseg/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


OSError: [Errno 28] No space left on device

In [41]:
results_df = pd.DataFrame(all_rows)
results_df.to_csv("grid_runs/summary_table.tsv", sep="\t", index=False)
results_df

,p_old,p_young,t_young_kya,t_old_kya,delta_kya,noem_old_precision,noem_old_recall,noem_old_f1,noem_young_precision,noem_young_recall,...,em_old_f1,em_young_precision,em_young_recall,em_young_f1,learned_lambda_n,learned_lambda_af,learned_lambda_old,learned_lambda_young,summary_json,out_dir
0,0.1,0.1,30000,35000,5000,0.5029,0.7774,0.6107,0.4881,0.2123,...,0.1871,0.4863,0.8758,0.6253,0.984283,0.095302,0.081148,0.030535,grid_runs/pold_0.1_pyoung_0.1_told_35000k_tyou...,grid_runs/pold_0.1_pyoung_0.1_told_35000k_tyou...
1,0.1,0.1,35000,40000,5000,0.4490,0.6424,0.5286,0.5802,0.3845,...,0.2575,0.5382,0.7524,0.6275,0.980678,0.095911,0.066724,0.029470,grid_runs/pold_0.1_pyoung_0.1_told_40000k_tyou...,grid_runs/pold_0.1_pyoung_0.1_told_40000k_tyou...
2,0.1,0.1,40000,45000,5000,0.4856,0.6100,0.5407,0.5813,0.4573,...,0.2743,0.5461,0.8464,0.6639,0.969394,0.096317,0.076757,0.032989,grid_runs/pold_0.1_pyoung_0.1_told_45000k_tyou...,grid_runs/pold_0.1_pyoung_0.1_told_45000k_tyou...
3,0.1,0.1,45000,50000,5000,0.4829,0.6621,0.5585,0.5952,0.4070,...,0.2892,0.5495,0.8243,0.6594,0.979030,0.094520,0.092577,0.038408,grid_runs/pold_0.1_pyoung_0.1_told_50000k_tyou...,grid_runs/pold_0.1_pyoung_0.1_told_50000k_tyou...
4,0.1,0.1,50000,55000,5000,0.4753,0.6053,0.5325,0.5714,0.4081,...,0.1889,0.5358,0.8609,0.6605,0.981834,0.095902,0.119426,0.043791,grid_runs/pold_0.1_pyoung_0.1_told_55000k_tyou...,grid_runs/pold_0.1_pyoung_0.1_told_55000k_tyou...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189,0.3,0.2,40000,45000,5000,0.5189,0.9552,0.6725,0.6852,0.0796,...,0.3632,0.5023,0.7869,0.6132,0.866044,0.096124,0.074523,0.030633,grid_runs/pold_0.3_pyoung_0.2_told_45000k_tyou...,grid_runs/pold_0.3_pyoung_0.2_told_45000k_tyou...
190,0.3,0.2,45000,50000,5000,0.5453,0.9758,0.6997,0.7162,0.0371,...,0.3537,0.4676,0.7961,0.5891,0.874295,0.096335,0.082704,0.035056,grid_runs/pold_0.3_pyoung_0.2_told_50000k_tyou...,grid_runs/pold_0.3_pyoung_0.2_told_50000k_tyou...
191,0.3,0.2,50000,55000,5000,0.5422,0.9624,0.6936,0.5060,0.0316,...,0.2830,0.4653,0.8644,0.6049,0.862658,0.098003,0.108270,0.041973,grid_runs/pold_0.3_pyoung_0.2_told_55000k_tyou...,grid_runs/pold_0.3_pyoung_0.2_told_55000k_tyou...
192,0.3,0.2,30000,40000,10000,0.5541,0.9366,0.6963,0.5786,0.0920,...,0.4509,0.4744,0.7101,0.5688,0.875682,0.095609,0.060387,0.027270,grid_runs/pold_0.3_pyoung_0.2_told_40000k_tyou...,grid_runs/pold_0.3_pyoung_0.2_told_40000k_tyou...


In [ ]:
# results = Parallel(n_jobs=N_JOBS)(
#     delayed(sims.process_one_chromosome)(seed, params, Ne, t, p_admix, OUTPUT_DIR)
#     for seed in random_seeds
# )
# print(f"All simulations are finished...All trees are in {OUTPUT_DIR}/")

# final_df = pd.concat(results, ignore_index=True)
# csv_path = os.path.join(OUTPUT_DIR, "all.tracts.summary.csv")
# final_df.to_csv(csv_path, index=False, sep='\t')

# print(f"Nd in EU tract table in {csv_path}")

In [ ]:
# # DAIseg — это готовый пайплайн/скрипт, который использует HMM внутри

# print(f"Running DAIseg in parallel for {len(random_seeds)} chromosomes...")
# # Run in parallel
# results = Parallel(n_jobs=N_JOBS)(
#     delayed(sims.run_daiseg_task)(seed, OUTPUT_DIR) 
#     for seed in random_seeds
# )

# valid_dfs = [res for res in results if res is not None]
# if valid_dfs:
#     final_df = pd.concat(valid_dfs, ignore_index=True)    
#     final_output = os.path.join(OUTPUT_DIR, "all.inferred.tracts.tsv")
#     final_df.to_csv(final_output, sep='\t', index=False)    
#     print(f"Done! Saved to {final_output}\n")
# else:
#     print("No tracts found.")

# print("Performance per Class without usin EM...\n\n")
# res = sims.calculate_class_metrics(
#     os.path.join(OUTPUT_DIR, "all.tracts.summary.csv"),
#     os.path.join(OUTPUT_DIR, "all.inferred.tracts.tsv"),
#     params['chrom_length']
#     )

# print(f"Total Analyzed Genome: {res['Total_BP']:,} bp")
# print("-" * 50)
# print(f"{'CLASS':<13} | {'PRECISION':<10} | {'RECALL':<7} | {'F1-SCORE':<10}")
# print("-" * 50)
# for cls, stats in res["Per_class"].items():
#     if cls == "Modern":
#         continue
#     print(f"{cls:<13} | {stats['Precision']:<10.2%} | {stats['Recall']:<7.2%} | {stats['F1']:<10.4f}")
# print("-" * 50)

In [ ]:
# ! python ../daiseg.py run.with.EM -jsons sims/config_seed_*.json -out ./sims/all.inferred.EM.tsv

In [ ]:
# gt_path = os.path.join(OUTPUT_DIR, "all.tracts.summary.csv")
# inf_path = os.path.join(OUTPUT_DIR, "all.inferred.EM.tsv")

# res = sims.calculate_class_metrics(gt_path, inf_path, params['chrom_length'])

# print(f"Total Analyzed Genome: {res['Total_BP']:,} bp")
# print("-" * 55)
# print(f"{'CLASS':<15} | {'PRECISION':<10} | {'RECALL':<10} | {'F1-SCORE':<10}")
# print("-" * 55)

# for cls in ["Archaic_old", "Archaic_young"]:
#     stats = res["Per_class"][cls]
#     print(f"{cls:<15} | {stats['Precision']:<10.2%} | {stats['Recall']:<10.2%} | {stats['F1']:<10.4f}")

# print("-" * 55)

In [ ]:
# gt_df = pd.read_csv(gt_path, sep='\t')
# inf_df = pd.read_csv(inf_path, sep='\t')

# cm, labels = sims.build_confusion_matrix(gt_df, inf_df, chrom_length=int(params['chrom_length']), window_size=1000)
# print(labels)
# print(cm)